# N02 Main Evaluation — Dataset Design & Preflight v1.0

**Goal:** freeze a controlled Main Evaluation corpus for the three N02 core observables: length-adjusted block-update norm, input–update cosine, and output/input norm ratio.

## Study design
- **100 semantic topic clusters × 4 matched text styles = 400 candidate texts.**
- Styles: general prose, question/dialogue, academic prose, and code + natural language.
- Length tiers: 34 short clusters, 33 medium clusters, 33 long clusters. Candidate token bands are 16–64, 65–128, and 129–256 tokens, respectively, and must be checked separately with each model tokenizer.
- Each `topic_id` has one rendition in each style. This matches content at a topic level, but the four renditions are repeated/matched observations, not four independent semantic units.
- Therefore, the planned confidence intervals for style contrasts resample **topic_id clusters** (10,000 bootstrap replicates), preserving all four renditions. Do not resample the 400 rows independently for matched-style contrasts.
- Pilot prompts are explicitly excluded.

**Preflight only:** this notebook audits the candidate text manifest. It does not load the language models or capture activations. No model activations may be examined until the final text manifest has passed the tokenizer/quality checks and its SHA-256 hash has been recorded.

The corpus is template-assisted and should receive human content/quality review. The code-mixed condition is a distinct format condition, not a pure isolated linguistic style; interpretation must acknowledge that code syntax itself differs from prose.


In [ ]:
%pip -q install -U transformers pandas matplotlib

## 1. Upload the candidate manifest

Download `N02_Main_Evaluation_Candidate_Manifest_v1.0.csv` with this notebook, then upload it when prompted. This is a **candidate** manifest, not yet frozen for confirmatory analysis.


In [ ]:
from pathlib import Path
import os, json, hashlib, re, math
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

manifest_path = Path('/content/N02_Main_Evaluation_Candidate_Manifest_v1.0.csv')
if not manifest_path.exists():
    from google.colab import files
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
    if not csv_names:
        raise FileNotFoundError('Upload N02_Main_Evaluation_Candidate_Manifest_v1.0.csv')
    manifest_path = Path('/content') / csv_names[0]

candidate_df = pd.read_csv(manifest_path)
print('Manifest:', manifest_path)
print('Rows:', len(candidate_df), 'Unique topic clusters:', candidate_df['topic_id'].nunique())
display(candidate_df.head(8))

## 2. Structural validation

A valid candidate must contain 100 topic clusters, exactly 4 unique styles per cluster, 400 rows total, no duplicate `text_id`, and no duplicated normalized text. These checks are necessary, not sufficient; content quality and token counts still need review.


In [ ]:
EXPECTED_STYLES = {
    'general_prose', 'question_dialogue', 'academic', 'code_plus_natural_language'
}
EXPECTED_TIER_COUNTS = {'short': 34, 'medium': 33, 'long': 33}
LENGTH_BANDS = {'short': (16, 64), 'medium': (65, 128), 'long': (129, 256)}

required_columns = {
    'topic_id', 'topic_domain', 'operation_id', 'length_tier', 'style',
    'text_id', 'text', 'role', 'pilot_excluded', 'sampling_unit'
}
missing_columns = required_columns - set(candidate_df.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

checks = {
    'rows_400': len(candidate_df) == 400,
    'topics_100': candidate_df['topic_id'].nunique() == 100,
    'text_id_unique': candidate_df['text_id'].is_unique,
    'nonempty_texts': candidate_df['text'].fillna('').str.strip().ne('').all(),
    'styles_exact': set(candidate_df['style']) == EXPECTED_STYLES,
    'no_duplicate_normalized_text': not candidate_df['text'].fillna('').str.lower().str.replace(r'\s+', ' ', regex=True).str.strip().duplicated().any(),
    'all_marked_pilot_excluded': candidate_df['pilot_excluded'].astype(str).str.lower().isin(['true','1']).all(),
}
cluster_style_counts = candidate_df.groupby('topic_id')['style'].nunique()
checks['four_styles_per_topic'] = len(cluster_style_counts) == 100 and cluster_style_counts.eq(4).all()
tier_topic_counts = candidate_df.drop_duplicates('topic_id')['length_tier'].value_counts().to_dict()
checks['tier_topic_counts_34_33_33'] = tier_topic_counts == EXPECTED_TIER_COUNTS

print('Structural checks:')
for name, passed in checks.items(): print(('PASS' if passed else 'FAIL'), name)
if not all(checks.values()):
    raise ValueError('Structural validation failed. Fix candidate manifest before tokenizer preflight.')
display(candidate_df.groupby(['length_tier','style']).size().unstack(fill_value=0))

## 3. Tokenizer-specific length audit

Run exact token counts separately for the two current N02 pilot model tokenizers: GPT-2 and Pythia-160M. A text only passes this preflight if its token count falls inside the assigned tier for **both** tokenizers. Count includes tokenizer-added special tokens as used in the audit below. Do not inspect activation values during text revision.


In [ ]:
from transformers import AutoTokenizer

TOKENIZER_IDS = {
    'gpt2': 'gpt2',
    'pythia-160m': 'EleutherAI/pythia-160m',
}
tokenizers = {key: AutoTokenizer.from_pretrained(model_id) for key, model_id in TOKENIZER_IDS.items()}

for model_key, tokenizer in tokenizers.items():
    candidate_df[f'tokens_{model_key}'] = candidate_df['text'].map(
        lambda text: len(tokenizer(text, add_special_tokens=True, truncation=False)['input_ids'])
    )
    low_high = candidate_df['length_tier'].map(LENGTH_BANDS)
    candidate_df[f'in_band_{model_key}'] = [
        lo <= int(n) <= hi for (lo, hi), n in zip(low_high, candidate_df[f'tokens_{model_key}'])
    ]

for model_key in TOKENIZER_IDS:
    col = f'tokens_{model_key}'
    ok_col = f'in_band_{model_key}'
    print(f'\n{model_key}: in-band {int(candidate_df[ok_col].sum())}/{len(candidate_df)}')
    display(candidate_df.loc[~candidate_df[ok_col], ['topic_id','length_tier','style','text_id',col,'text']].head(25))

length_summary = candidate_df.groupby(['length_tier','style']).agg(
    count=('text_id','count'),
    gpt2_median=('tokens_gpt2','median'),
    gpt2_min=('tokens_gpt2','min'),
    gpt2_max=('tokens_gpt2','max'),
    pythia_median=('tokens_pythia-160m','median'),
    pythia_min=('tokens_pythia-160m','min'),
    pythia_max=('tokens_pythia-160m','max'),
).reset_index()
display(length_summary)

# Topic-level within-matched-set spread; warnings are for pre-activation revision.
spread_rows = []
for topic_id, group in candidate_df.groupby('topic_id'):
    row = {'topic_id': topic_id, 'length_tier': group['length_tier'].iloc[0]}
    for model_key in TOKENIZER_IDS:
        counts = group[f'tokens_{model_key}'].astype(float)
        row[f'{model_key}_token_min'] = counts.min()
        row[f'{model_key}_token_max'] = counts.max()
        row[f'{model_key}_token_ratio_max_min'] = counts.max() / max(counts.min(), 1)
        row[f'{model_key}_token_spread'] = counts.max() - counts.min()
    spread_rows.append(row)
spread_df = pd.DataFrame(spread_rows)
print('Topic groups with >1.8x max/min token spread (review before freeze):')
display(spread_df[(spread_df['gpt2_token_ratio_max_min'] > 1.8) | (spread_df['pythia-160m_token_ratio_max_min'] > 1.8)].head(30))

## 4. Revision gate and freeze

Before any activation capture:
1. Review text quality and semantic matching across each four-style topic cluster.
2. Revise any text outside its assigned token band using only prompt text/tokenizer counts (do not look at model activations first).
3. Inspect topic-level token spread and the length summaries; match/revise texts with large within-topic spread where feasible.
4. Save the finalized manifest and hash it. The confirmatory run must read the frozen file and not silently regenerate or edit it.

The cell below **will not declare the manifest frozen if any candidate is outside its assigned tier for either tokenizer**. It creates a preflight report regardless. The semantic/content review checkbox is an explicit human attestation; leave it `False` until that review is done.


In [ ]:
CONTENT_REVIEW_COMPLETE = False  # Change to True only after human review of all topic-matched quartets.

band_failures = int((~candidate_df['in_band_gpt2']).sum() + (~candidate_df['in_band_pythia-160m']).sum())
all_bands_pass = candidate_df['in_band_gpt2'].all() and candidate_df['in_band_pythia-160m'].all()
all_styles_per_topic = candidate_df.groupby('topic_id')['style'].nunique().eq(4).all()
freeze_allowed = bool(all(checks.values()) and all_bands_pass and all_styles_per_topic and CONTENT_REVIEW_COMPLETE)

# Always write diagnostics, even when freeze is blocked.
OUT_DIR = Path('/content/n02_main_eval_preflight')
OUT_DIR.mkdir(parents=True, exist_ok=True)
candidate_df.to_csv(OUT_DIR / 'N02_candidate_manifest_with_token_counts.csv', index=False)
length_summary.to_csv(OUT_DIR / 'N02_token_length_summary.csv', index=False)
spread_df.to_csv(OUT_DIR / 'N02_topic_token_spread.csv', index=False)

manifest_sha256 = hashlib.sha256(candidate_df['text_id'].astype(str).str.cat(candidate_df['text'].astype(str), sep='\n').encode('utf-8')).hexdigest()
report = {
    'protocol_version': 'N02_Main_Evaluation_Dataset_Design_v1.0',
    'rows': int(len(candidate_df)),
    'topic_clusters': int(candidate_df['topic_id'].nunique()),
    'styles_per_topic': 4,
    'tokenizers': TOKENIZER_IDS,
    'length_bands': LENGTH_BANDS,
    'band_failures_across_tokenizers': band_failures,
    'all_candidates_in_band_for_both_tokenizers': bool(all_bands_pass),
    'human_content_review_complete': bool(CONTENT_REVIEW_COMPLETE),
    'frozen_for_confirmatory_analysis': freeze_allowed,
    'candidate_text_manifest_sha256': manifest_sha256,
    'resampling_unit_for_matched_style_contrasts': 'topic_id cluster; retain all four styles per resampled topic',
    'bootstrap_replicates': 10000,
    'pilot_prompts_excluded': True,
    'notes': 'Do not capture activations unless frozen_for_confirmatory_analysis is true. The code-mixed condition is a format condition as well as a style condition.'
}
with open(OUT_DIR / 'N02_preflight_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2, ensure_ascii=False))

if freeze_allowed:
    frozen_path = OUT_DIR / 'N02_Main_Evaluation_FROZEN_Manifest.csv'
    candidate_df.to_csv(frozen_path, index=False)
    frozen_file_sha256 = hashlib.sha256(frozen_path.read_bytes()).hexdigest()
    report['frozen_manifest_file_sha256'] = frozen_file_sha256
    report['frozen_manifest_path'] = str(frozen_path)
    with open(OUT_DIR / 'N02_preflight_report.json', 'w', encoding='utf-8') as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    print('FROZEN MANIFEST:', frozen_path)
    print('FROZEN FILE SHA256:', frozen_file_sha256)
else:
    print('NOT FROZEN: resolve token-band failures and complete content review before activation capture.')

## 5. Pre-registered analysis outline (after freeze only)

**Core observables per text × model × block**
- `update_norm_per_token = ||h_(l+1) - h_l||_F / sqrt(T)` (keep raw Frobenius norm as QC).
- `input_update_cosine = cos(h_l, Δh_l)`; compute from tensors when available.
- `output_input_norm_ratio = ||h_(l+1)||_F / ||h_l||_F`.

**Primary exploratory-confirmatory contrast framework**
- Keep GPT-2 and Pythia analyses separate first, then treat model as a factor only in a prespecified joint model because hidden-state scales differ.
- Within each `topic_id`, compare styles while controlling actual tokenizer-specific token count and length tier.
- Layer depth is aligned by normalized block position `rho = block_idx / max(L - 1, 1)` for block indices. Preserve raw block index too.
- Use a topic-cluster bootstrap with 10,000 replicates, resampling 100 topic IDs and retaining all four style variants inside each selected topic.
- For layer-wise familywise inference, use a preregistered max-statistic permutation or explicitly limit confirmatory claims to prespecified layer summaries; use Holm correction across the three core observables if testing each as a co-primary outcome.
- Do not claim chaos, physical energy, thermodynamic temperature, or a phase transition from these geometric observables alone.

The N02 pilot baseline is in the separate file `N02_Pilot_Baseline_v1.0.md`; it remains exploratory and is not part of the Main Evaluation dataset.
